In [0]:
from pyspark.sql.functions import col, split, explode, expr, monotonically_increasing_id

In [0]:
chicago = spark.table("silver_chicago")
dallas = spark.table("silver_dallas")

In [0]:
chicago_violations = chicago.withColumn(
    "violation",
    explode(split(col("violations"), "\\|"))
)

chicago_violations = chicago_violations.select(
    "inspection_id",
    "violation"
)

In [0]:
dallas = dallas.withColumn("inspection_id", monotonically_increasing_id())

In [0]:
dallas_violations = dallas.selectExpr(
    "inspection_id",
    "stack(3, `violation_memo_-_1`, `violation_memo_-_2`, `violation_memo_-_3`) as violation"
)

In [0]:
chicago_violations = chicago_violations.filter(col("violation").isNotNull())
dallas_violations = dallas_violations.filter(col("violation").isNotNull())

In [0]:
final_violations = chicago_violations.unionByName(dallas_violations)

In [0]:
final_violations.printSchema()
final_violations.show(5)

In [0]:
final_violations.write.mode("overwrite").saveAsTable("fact_violation")